In [0]:
%pip install geopandas rasterio shapely

In [0]:
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping
import tempfile
import shutil

In [0]:
import tempfile
import shutil
import os
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping

flight_list = dbutils.jobs.taskValues.get(
    taskKey="3_orquestator", 
    key="missing_clips", 
    default=[]
)

if not flight_list:
    dbutils.notebook.exit("No pending flights to process. Exiting gracefully.")

print(f" Received {len(flight_list)} flights for plot cropping.\n")

for flight_path in flight_list:
    print("-" * 60)
    print(f" PROCESSING FLIGHT: {flight_path}")
    
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
        
    base_dir = os.path.dirname(flight_path)
    parent_dir = os.path.dirname(base_dir)
    field_data_dir = os.path.join(parent_dir, "field_data")

    possible_ortho_names = ["RGB.tif", "MS.tif"]
    rgb_path = None

    for name in possible_ortho_names:
            candidate = f"{base_dir}/*/*/*/{name}"
            matches = glob.glob(candidate)
            if matches:
                rgb_path = matches[0]
                break

    out_dir = os.path.join(base_dir) 
    
    print(f" Searching geometries in: {field_data_dir}")
    vector_path = os.path.join(field_data_dir, "location_boundary.geojson")
    
    if rgb_path is None:
        print(f" Error: No RGB.tif or MS.tif found in {base_dir}. Skipping flight...")
        continue
    else:
        ortho_name = os.path.splitext(os.path.basename(rgb_path))[0]
        print(f" Orthomosaic found: {os.path.basename(rgb_path)}")
        
    if not os.path.exists(vector_path):
       print(f" Error: No 'location_boundary.geojson' found in {field_data_dir}. Skipping flight...") 
       continue 
        
    print(f" Vector file found: {os.path.basename(vector_path)}")

    os.makedirs(out_dir, exist_ok=True)

    # Local scratch directory on the cluster's own disk
    local_tmp_dir = tempfile.mkdtemp(prefix="plot_clip_")

    try:
        print(" Loading geometries and checking Coordinate Reference Systems (CRS)...")
        gdf_plots = gpd.read_file(vector_path)
        
        with rasterio.open(rgb_path) as src:
            raster_crs = src.crs
            
            if gdf_plots.crs != raster_crs:
                print(f" Reprojecting polygons from {gdf_plots.crs} to {raster_crs}...")
                gdf_plots = gdf_plots.to_crs(raster_crs)

            n_bands = src.count
            if n_bands <= 4:
                out_driver = "PNG"
                out_ext = "png"
            else:
                out_driver = "GTiff"
                out_ext = "tif"
            print(f" Detected {n_bands} bands -> using {out_driver} for plot outputs ({out_ext}).")

            # =====================================================================
            # 1. CLIP AND SAVE THE ORIGINAL .TIF WITH THE FULL BOUNDARY
            # =====================================================================
            print(" Clipping the original .tif file based on the full boundary...")
            all_geometries = [mapping(geom) for geom in gdf_plots.geometry]
            out_image_full, out_transform_full = mask(src, all_geometries, crop=True)
            
            out_meta_full = src.meta.copy()
            out_meta_full.update({
                "driver": "GTiff",
                "height": out_image_full.shape[1],
                "width": out_image_full.shape[2],
                "transform": out_transform_full,
                "compress": "lzw" # Recommended compression for TIFs
            })
            
            # Saved with the _clipped suffix to avoid corrupting the file currently being read
            clipped_tif_name = f"{ortho_name}_clipped.tif"
            local_full_tif_path = os.path.join(local_tmp_dir, clipped_tif_name)
            
            with rasterio.open(local_full_tif_path, "w", **out_meta_full) as dest:
                dest.write(out_image_full)
                
            final_full_tif_path = os.path.join(out_dir, clipped_tif_name)
            shutil.copyfile(local_full_tif_path, final_full_tif_path)
            
            # OPTIONAL: If you want to replace/delete the original TIF file, uncomment the following line:
            # os.replace(final_full_tif_path, rgb_path)

            # =====================================================================
            # 2. CLIP AND SAVE EACH INDIVIDUAL POLYGON (Original behavior)
            # =====================================================================
            print(f" Clipping {len(gdf_plots)} detected plots...")
            
            for idx, row in gdf_plots.iterrows():
                geometry = [mapping(row.geometry)]
                
                out_image, out_transform = mask(src, geometry, crop=True)
                
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": out_driver,  
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                })
                
                # ADDED _plot_{idx} TO AVOID OVERWRITING FILES
                file_name = f"{ortho_name}_plot_{idx}.{out_ext}"
                
                # Write to LOCAL disk first 
                local_out_path = os.path.join(local_tmp_dir, file_name)
                with rasterio.open(local_out_path, "w", **out_meta) as dest:
                    dest.write(out_image)
                
                # Then copy the finished file to the Volume
                final_out_path = os.path.join(out_dir, file_name)
                shutil.copyfile(local_out_path, final_out_path)
                    
            print(f" SUCCESS: Full clipped TIF and {len(gdf_plots)} images saved to {out_dir}")

    except Exception as e:
        print(f" An error occurred processing this flight: {e}")
    
    finally:
        # Clean up local scratch files regardless of success/failure
        shutil.rmtree(local_tmp_dir, ignore_errors=True)

print("\n" + "="*70)
print(" LOCATION CROPPING PIPELINE FINISHED SUCCESSFULLY.")